# SHA-256 Markov Carry Topology Verification v1

This is a new companion notebook for the pasted carry-scar synthesis.

It verifies the carry-chain theorem stack and patches the dangerous overclaims.

Core recurrence:

$$
q_{j+1}=\left\lfloor\frac{q_j+n_j}{2}\right\rfloor,\qquad
n_j\sim\mathrm{Binomial}(k,1/2)
$$

Core scar readout:

$$
S_j=q_j\bmod 2.
$$

Scope:

1. Build $T^{(k)}$.
2. Verify Eulerian stationary law.
3. Verify $\operatorname{spec}(T^{(k)})=\{2^{-m}\}$.
4. Verify $\Sigma(k)$.
5. Fit parity-selected modal coefficients.
6. Audit corrected $c_1(2n)$ values.
7. Flag publication guardrails.


In [ ]:
# Imports and settings

from fractions import Fraction
from math import comb, factorial, cos, sin, pi
from typing import List, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MAX_K = 16
MAX_J = 16

print("Carry topology verification notebook initialized.")


## 1. Carry transition matrix

For $k$ operands:

$$
q_j\in\{0,\dots,k-1\}
$$

and:

$$
q_{j+1}=\left\lfloor\frac{q_j+n_j}{2}\right\rfloor.
$$

This cell builds row-stochastic $T$ with:

$$
p_{j+1}=p_jT.
$$


In [ ]:
def carry_transition_fraction(k: int):
    T = [[Fraction(0, 1) for _ in range(k)] for _ in range(k)]
    denom = 2 ** k
    for q in range(k):
        for n in range(k + 1):
            qn = (q + n) // 2
            if 0 <= qn < k:
                T[q][qn] += Fraction(comb(k, n), denom)
    return T

def carry_transition_float(k: int) -> np.ndarray:
    return np.array([[float(x) for x in row] for row in carry_transition_fraction(k)], dtype=float)

for k in range(2, 7):
    T = carry_transition_float(k)
    print(f"k={k}, row sums =", np.round(T.sum(axis=1), 8))


## 2. Eulerian stationary law

Stationary law:

$$
\pi_q=\frac{A(k,q)}{k!}
$$

and stationary clean-scar probability:

$$
P_\infty^{(k)}(S=0)=\sum_{q\ \mathrm{even}}\frac{A(k,q)}{k!}.
$$


In [ ]:
def eulerian_number(n: int, m: int) -> int:
    if m < 0 or m >= n:
        return 0
    A = [[0 for _ in range(n)] for __ in range(n + 1)]
    A[1][0] = 1
    for nn in range(2, n + 1):
        for mm in range(nn):
            left = (nn - mm) * (A[nn-1][mm-1] if mm-1 >= 0 else 0)
            right = (mm + 1) * (A[nn-1][mm] if mm < nn-1 else 0)
            A[nn][mm] = left + right
    return A[n][m]

def eulerian_stationary(k: int):
    return [Fraction(eulerian_number(k, q), factorial(k)) for q in range(k)]

def stationary_even_prob(k: int) -> Fraction:
    pi_vec = eulerian_stationary(k)
    return sum(pi_vec[q] for q in range(k) if q % 2 == 0)

rows = []
for k in range(1, 11):
    rows.append({
        "k": k,
        "pi": [str(x) for x in eulerian_stationary(k)],
        "P_inf_even": str(stationary_even_prob(k)),
        "decimal": float(stationary_even_prob(k)),
    })

pd.DataFrame(rows)


In [ ]:
verify_rows = []
for k in range(2, 11):
    T = carry_transition_float(k)
    pi = np.array([float(x) for x in eulerian_stationary(k)])
    verify_rows.append({
        "k": k,
        "max_abs_residual_piT_minus_pi": np.max(np.abs(pi @ T - pi)),
    })
pd.DataFrame(verify_rows)


## 3. Spectrum verification

Claim:

$$
\operatorname{spec}(T^{(k)})=\{1,1/2,1/4,\dots,2^{-(k-1)}\}.
$$


In [ ]:
spec_rows = []
for k in range(2, 13):
    T = carry_transition_float(k)
    eig = np.linalg.eigvals(T)
    eig_sorted = sorted([float(np.real_if_close(x).real) for x in eig], reverse=True)
    target = [2 ** (-m) for m in range(k)]
    spec_rows.append({
        "k": k,
        "computed": [round(x, 10) for x in eig_sorted],
        "target": [round(x, 10) for x in target],
        "max_abs_error": max(abs(a-b) for a, b in zip(eig_sorted, target)),
    })
pd.DataFrame(spec_rows)


## 4. First-bit spectroscopy $\Sigma(k)$

Closed form:

$$
\Sigma(k)
=
\frac12+
2^{-(k/2+1)}
\left[
\cos\left(\frac{k\pi}{4}\right)+
\sin\left(\frac{k\pi}{4}\right)
\right].
$$


In [ ]:
def scar_clean_probability_direct(k: int, j: int) -> Fraction:
    T = carry_transition_fraction(k)
    p = [Fraction(0, 1) for _ in range(k)]
    p[0] = Fraction(1, 1)
    for _ in range(j):
        p_next = [Fraction(0, 1) for _ in range(k)]
        for q in range(k):
            for qp in range(k):
                p_next[qp] += p[q] * T[q][qp]
        p = p_next
    return sum(p[q] for q in range(k) if q % 2 == 0)

def sigma_formula(k: int) -> float:
    return 0.5 + (2 ** (-(k/2 + 1))) * (cos(k*pi/4) + sin(k*pi/4))

sigma_rows = []
for k in range(1, 17):
    direct = scar_clean_probability_direct(k, 1)
    formula = sigma_formula(k)
    sigma_rows.append({
        "k": k,
        "Sigma_direct_exact": str(direct),
        "Sigma_direct": float(direct),
        "Sigma_formula": formula,
        "abs_error": abs(float(direct) - formula),
    })
pd.DataFrame(sigma_rows)


## 5. Decay curves

Directly compute:

$$
P(S_j^{(k)}=0).
$$

The LSB anchor is:

$$
P(S_0^{(k)}=0)=1.
$$


In [ ]:
decay_rows = []
for k in range(2, 9):
    for j in range(0, MAX_J + 1):
        p = scar_clean_probability_direct(k, j)
        decay_rows.append({"k": k, "j": j, "P_clean": float(p), "P_exact": str(p)})
decay_df = pd.DataFrame(decay_rows)

plt.figure(figsize=(10, 5))
for k in range(2, 9):
    sub = decay_df[decay_df["k"] == k]
    plt.plot(sub["j"], sub["P_clean"], marker="o", label=f"k={k}")
plt.axhline(0.5, linestyle="--")
plt.title(r"Carry scar clean probability $P(S_j^{(k)}=0)$")
plt.xlabel("bit position j")
plt.ylabel("P clean")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

decay_df.head(20)


## 6. Parity-selected modal expansion

Allowed non-stationary modes satisfy:

$$
m\equiv k-1\pmod2.
$$

Fit:

$$
P(S_j^{(k)}=0)
=
P_\infty^{(k)}
+
\sum_{m}c_m(2^{-m})^j.
$$


In [ ]:
def allowed_modes(k: int):
    return [m for m in range(1, k) if (m - (k - 1)) % 2 == 0]

def fit_modal_coefficients(k: int):
    modes = allowed_modes(k)
    p_inf = float(stationary_even_prob(k))
    js = list(range(len(modes)))
    A = np.array([[2 ** (-m * j) for m in modes] for j in js], dtype=float)
    y = np.array([float(scar_clean_probability_direct(k, j)) - p_inf for j in js], dtype=float)
    coeff = np.linalg.solve(A, y)
    return {m: coeff[i] for i, m in enumerate(modes)}

modal_rows = []
for k in range(2, 13):
    coeffs = fit_modal_coefficients(k)
    for m, c in coeffs.items():
        modal_rows.append({"k": k, "mode_m": m, "eigenvalue": 2**(-m), "coefficient": c})
modal_df = pd.DataFrame(modal_rows)
modal_df


In [ ]:
recon_rows = []
for k in range(2, 13):
    coeffs = fit_modal_coefficients(k)
    p_inf = float(stationary_even_prob(k))
    for j in range(0, 12):
        pred = p_inf + sum(c * (2 ** (-m)) ** j for m, c in coeffs.items())
        direct = float(scar_clean_probability_direct(k, j))
        recon_rows.append({"k": k, "j": j, "direct": direct, "modal_pred": pred, "abs_error": abs(pred-direct)})
recon_df = pd.DataFrame(recon_rows)
recon_df.groupby("k")["abs_error"].max().reset_index()


## 7. $c_1(2n)$ audit

Correct exact values:

$$
\frac12,\ -\frac12,\ \frac13,\ -\frac{17}{90},\ \frac{31}{315},\ -\frac{691}{14175},\ \frac{10922}{467775},\ -\frac{929569}{85135050}.
$$

Publication patch:

$$
-\frac{473}{9703},\quad \frac{216}{9251},\quad -\frac{41}{3755}
$$

are approximants, not exact sequence terms.


In [ ]:
correct_c1 = {
    2: Fraction(1, 2),
    4: Fraction(-1, 2),
    6: Fraction(1, 3),
    8: Fraction(-17, 90),
    10: Fraction(31, 315),
    12: Fraction(-691, 14175),
    14: Fraction(10922, 467775),
    16: Fraction(-929569, 85135050),
}

bad_approximants = {
    12: Fraction(-473, 9703),
    14: Fraction(216, 9251),
    16: Fraction(-41, 3755),
}

audit_rows = []
for k in sorted(correct_c1):
    coeffs = fit_modal_coefficients(k)
    c1_fit = coeffs.get(1, np.nan)
    exact = correct_c1[k]
    row = {
        "k": k,
        "c1_fit_numeric": c1_fit,
        "correct_exact": str(exact),
        "correct_decimal": float(exact),
        "fit_minus_correct": c1_fit - float(exact),
        "sign_formula": (-1) ** (k//2 + 1),
        "sign_actual": 1 if exact > 0 else -1,
    }
    if k in bad_approximants:
        bad = bad_approximants[k]
        row["bad_approximant"] = str(bad)
        row["bad_decimal"] = float(bad)
        row["bad_minus_correct"] = float(bad - exact)
    audit_rows.append(row)

audit_df = pd.DataFrame(audit_rows)
audit_df


In [ ]:
ks = np.array(sorted(correct_c1.keys()))
vals = np.array([abs(float(correct_c1[k])) for k in ks])

plt.figure(figsize=(8, 4))
plt.semilogy(ks, vals, marker="o")
plt.title(r"Corrected even-$k$ slow-mode magnitude $|c_1(k)|$")
plt.xlabel("k")
plt.ylabel(r"$|c_1(k)|$ log scale")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

ratio_rows = []
for a, b in zip(ks[:-1], ks[1:]):
    ratio_rows.append({
        "from_k": int(a),
        "to_k": int(b),
        "ratio_abs_next_over_current": abs(float(correct_c1[int(b)])) / abs(float(correct_c1[int(a)])),
    })
pd.DataFrame(ratio_rows)


## 8. Publication guardrails

### Guard A — $k_{\rm eff}=3$ shift windows

A missing `SHR` component inside $\sigma_0$ or $\sigma_1$ does **not automatically** reduce the modular addition operand count from $4$ to $3$.

The schedule addition still has four addends:

$$
W_t=\sigma_1(W_{t-2})+W_{t-7}+\sigma_0(W_{t-15})+W_{t-16}.
$$

A structural zero inside $\sigma$ changes bias/correlation structure of one addend. It is literal $k=3$ only when an entire addend bit is fixed to zero.

### Guard B — $c_1$ asymptotics

The $|c_1(2n)|$ trend is strong but should remain an empirical asymptotic unless an analytic derivation is included.

### Guard C — hardness wall

“Rounds 1–6 transparent / round 7 wall” must be framed as an experimental constraint-solving phase transition unless the exact solver model and benchmark are specified.

### Guard D — inversion claims

Carry-scar spectroscopy supplies anchors, priors, and scoring fields. It is not a generic digest-only preimage solver.


# Final Ψ-collapse

Verified:

$$
\boxed{
\pi_q=\frac{A(k,q)}{k!}
}
$$

$$
\boxed{
P_\infty^{(k)}(S=0)=\sum_{q\ even}\frac{A(k,q)}{k!}
}
$$

$$
\boxed{
\operatorname{spec}(T^{(k)})=\{2^{-m}:m=0,\dots,k-1\}
}
$$

$$
\boxed{
\Sigma(k)=
\frac12+
2^{-(k/2+1)}
\left[
\cos\left(\frac{k\pi}{4}\right)+
\sin\left(\frac{k\pi}{4}\right)
\right]
}
$$

Corrected:

$$
\boxed{
c_1(12)=-\frac{691}{14175},\quad
c_1(14)=\frac{10922}{467775},\quad
c_1(16)=-\frac{929569}{85135050}.
}
$$

Next notebook rail:

$$
\boxed{
\text{Transfer Grammar v3: monotone ramps and boundary curvature.}
}
$$
